# 10차시 강사용 Notebook — 모델 평가와 새 데이터 예측

**시연 전 안내**: Orange3 Predictions 위젯으로 같은 새 로트를 먼저 예측해본 뒤 이 노트북으로 넘어갑니다.

## 1단계. 9차시 모델 다시 만들기
**설명 포인트**: random_state가 같으면 언제 다시 실행해도 같은 모델이 만들어진다는 점을 강조.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("../../data/weekly/week09/week09_pass_fail_train.csv").dropna()
feature_cols = ["온도_섭씨", "압력_Pa", "가스유량_slm", "두께_nm", "진공도_mTorr", "습도_pct"]
X = df[feature_cols]
y = df["검사결과"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",4
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current no

## 2단계. 새 로트 불러오기

In [2]:
new_lots = pd.read_csv("../../data/weekly/week09/week09_new_lots_to_predict.csv")
X_new = new_lots[feature_cols]
print(new_lots.shape)

(9, 8)


## 3단계. 새 로트 예측하기

In [3]:
new_lots["예측_검사결과"] = model.predict(X_new)
print(new_lots[["공정명", "설비번호", "온도_섭씨", "예측_검사결과"]])

  공정명   설비번호  온도_섭씨  예측_검사결과
0  식각  EQ-02  294.5        0
1  산화  EQ-02  301.5        0
2  식각  EQ-04  289.8        0
3  포토  EQ-04  295.6        0
4  식각  EQ-04  299.1        0
5  포토  EQ-03  302.0        0
6  산화  EQ-01  305.2        0
7  산화  EQ-03  299.1        0
8  세정  EQ-01  300.6        0


## 4단계. 확률까지 확인하기
**설명 포인트**: 모두 '합격'이어도 확률이 다르다는 것을 강조 — 0번, 6번 로트가 상대적으로 위험.

In [4]:
proba = model.predict_proba(X_new)
new_lots["불합격_확률"] = proba[:, 1].round(3)

print(new_lots[["공정명", "설비번호", "온도_섭씨", "예측_검사결과", "불합격_확률"]])

  공정명   설비번호  온도_섭씨  예측_검사결과  불합격_확률
0  식각  EQ-02  294.5        0   0.364
1  산화  EQ-02  301.5        0   0.067
2  식각  EQ-04  289.8        0   0.158
3  포토  EQ-04  295.6        0   0.067
4  식각  EQ-04  299.1        0   0.067
5  포토  EQ-03  302.0        0   0.067
6  산화  EQ-01  305.2        0   0.364
7  산화  EQ-03  299.1        0   0.067
8  세정  EQ-01  300.6        0   0.067


## 5단계. 가장 위험한 로트 찾기

In [5]:
riskiest = new_lots.sort_values("불합격_확률", ascending=False).head(2)
print(riskiest[["공정명", "설비번호", "온도_섭씨", "불합격_확률"]])

  공정명   설비번호  온도_섭씨  불합격_확률
0  식각  EQ-02  294.5   0.364
6  산화  EQ-01  305.2   0.364


## 6단계. 오류 대처 방법
- `ValueError: columns are missing` 발생 시: feature_cols 순서/이름이 학습 때와 같은지 확인.

## 확장 실습(빠른 학습자용)
max_depth를 바꿔 다시 학습시킨 뒤 같은 새 로트를 예측해 결과가 달라지는지 비교해보게 한다.

In [6]:
model2 = DecisionTreeClassifier(max_depth=8, random_state=42)
model2.fit(X_train, y_train)
print(model2.predict(X_new))

[1 0 0 0 0 0 0 0 0]


## 7단계(회고). 10차시 되짚기

**진행 방법**: 학생들과 함께 1~9차시 표(강의 자료 참고)를 다시 훑어보며, 각자 가장 인상 깊었던 차시를 한 명씩 짧게 공유하게 한다.

## 7단계. AI에게 질문하며 더 알아보기
**설명 포인트**: 10차시 마지막 실습이다. data drift, 모델 저장 같은 질문은 "이 과정 이후
실무에서 마주칠 다음 단계"로 자연스럽게 이어 설명해주면 좋은 마무리가 된다.

질문 예시:
- "predict()와 predict_proba()는 내부적으로 어떻게 다른 계산을 하나요?"
- "학습시킨 모델을 저장했다가 나중에 다시 불러와서 쓰려면 어떻게 하나요?"
- "새 데이터가 학습 데이터 범위를 벗어나면 예측이 부정확해질 수 있다는데, 왜 그런가요?"
- "시간이 지나며 모델 성능이 떨어지는 현상을 뭐라고 부르나요?"

## 8단계. AI에게 코드 생성 요청하고 직접 실행해보기
**설명 포인트**: `zip()`, `sorted(..., key=lambda ...)`는 오늘까지 배우지 않은 문법이다 —
10차시 마지막이므로 "이제 스스로 AI에게 설명을 요청해 해석해보는 것"도 훈련으로 안내한다.

프롬프트 예시: "로트 이름과 불합격 확률 리스트를 받아서, 확률이 높은 순서로 정렬하고 위험도
등급(높음/보통/낮음)을 함께 표시하는 코드를 만들어줘."

아래는 AI가 생성해줄 수 있는 예시 코드와 실행 결과다.

In [ ]:
labels = new_lots["공정명"] + "-" + new_lots["설비번호"]

for lot, p in sorted(zip(labels, new_lots["불합격_확률"]), key=lambda x: x[1], reverse=True):
    if p >= 0.3:
        risk = "높음"
    elif p >= 0.15:
        risk = "보통"
    else:
        risk = "낮음"
    print(f"{lot}: 불합격 확률 {p:.1%} → 위험도 {risk}")

## 9단계. 연습문제 — 위험 로트 개수 세기
**설명 포인트**: 4차시에서 배운 불리언 인덱싱(조건 검색) `df[조건]`을 오늘 새로 만든 열
(`불합격_확률`)에 그대로 적용해보는 단계다. 10차시 마지막에 4차시 스킬로 다시 돌아오는
좋은 순환 복습이 된다 — "새로운 걸 배우지 않아도, 배운 도구를 새 데이터에 다시 쓸 수
있다"는 메시지를 전달하기 좋다.

In [ ]:
risky_lots = new_lots[new_lots["불합격_확률"] > 0.5]
print(f"위험 로트(불합격 확률 50% 초과) 개수: {len(risky_lots)}개")

## 10단계. AI로 재미있는 미니 프로그램 만들기 🎉 — 10차시 완주 인증서 만들기
**설명 포인트**: 10차시, 그리고 10주 전체 과정의 마지막 실습이다. 정확한 코드보다 "완주를
자축하는 경험" 자체가 목적이므로, 시간이 허락하면 학생마다 자유롭게 꾸민 수료증 문구를
한 명씩 발표시키며 과정을 마무리하면 좋다.

프롬프트 예시: "input()으로 이름을 입력받고, 이모지나 별표(*)로 테두리를 두른 '10차시
반도체 데이터 분석 과정 수료증'을 이름과 함께 예쁘게 출력하는 파이썬 코드를 만들어줘."

아래는 AI가 생성해줄 수 있는 예시 코드다(자동 실행 검증에서는 `input()` 대신 예시 값을
대입).

In [ ]:
name = "지민"  # 실제로는 input("이름을 입력하세요: ")

print("🎓" * 15)
print("   10차시 반도체 데이터 분석 과정 수료증")
print("   위 사람은 10차시 전체 과정을 성실히 마쳤음을 증명합니다.")
print(f"   이름: {name}")
print("🎓" * 15)